# **ENSO Phase Lifetime under Volcanic Forcing - Models**

Outputs: 

1) Per-eruption raw remaining lifetime CSV
2) Control (non-eruption) remaining lifetime CSV

In [4]:
import os
import numpy as np
import pandas as pd
import xarray as xr
from xarray import CFTimeIndex
 
 
# USER SETTINGS
 
MODEL_PATHS = {
    "BCC-CSM1-1":         {"path": "/g/data/ob22/jxb548/PMIPDATA/past1000/ts_bcc-csm1-1_r1i1p1_0850_2000_r240x120.nc", "year_offset": 0},
    "CCSM4":              {"path": "/g/data/ob22/jxb548/PMIPDATA/past1000/ts_CCSM4_r1i1p1_0850_1850_r240x120.nc", "year_offset": 0},
    "CSIRO-Mk3L-1-2":     {"path": "/g/data/ob22/jxb548/PMIPDATA/past1000/ts_CSIRO-Mk3L-1-2_r1i1p1_0851_1850_r240x120.nc", "year_offset": 0},
    "GISS-E2-R":          {"path": "/g/data/ob22/jxb548/PMIPDATA/past1000/ts_GISS-E2-R_r1i1p121_0850_1850_r240x120.nc", "year_offset": 0},
    "IPSL-CM5A-LR":       {"path": "/g/data/ob22/jxb548/PMIPDATA/past1000/ts_IPSL-CM5A-LR_r1i1p1_0850_1850_r240x120.nc", "year_offset": 0},
    "MIROC-ES2L":         {"path": "/g/data/ob22/jxb548/PMIPDATA/past1000/ts_MIROC-ES2L_r1i1p1_0850_1849_r240x120.nc", "year_offset": 0},
    "MIROC-ESM":          {"path": "/g/data/ob22/jxb548/PMIPDATA/past1000/ts_MIROC-ESM_r1i1p1_0850_1849_r240x120.nc", "year_offset": 0},
    "MPI-ESM-P":          {"path": "/g/data/ob22/jxb548/PMIPDATA/past1000/ts_MPI-ESM-P_r1i1p1_0850_1849_r240x120.nc", "year_offset": 0},
    "MPI-ESM1-2":         {"path": "/g/data/ob22/ft3359/PMIPDATA/ts_Amon_MPI-ESM1-2-LR_past2k_r1i1p1f1_gn_700101-885012_regridded.nc", "year_offset": 6151},
    "MRI-ESM2-0":         {"path": "/g/data/ob22/ft3359/PMIPDATA/ts_Amon_MRI-ESM2-0_past1000_r1i1p1f1_gn_085001-184912_regridded.nc", "year_offset": 0} 
    
}
 
# ERUPTION LIST: all 20 eruptions (850-1849 CE) with known seasonality.

ERUPTIONS_RAW = [
    (939.0,  4.0,  1.0,  63.6, -1.0, 16.23, 4.97),
    (946.0, 11.0,  1.0,  42.0, -1.0,  1.72, 0.61),
    (1257.0, 7.0,  1.0,  -8.4,  1.4, 59.42, 10.86),
    (1477.0, 2.0,  1.0,  64.6, -1.0,  5.12, 1.61),
    (1510.0, 7.0, 25.0,  64.0, -1.0,  2.30, 0.83),
    (1585.0, 1.0, 10.0,  19.5, 10.6,  8.51, 2.34),
    (1595.0, 3.0,  1.0,   4.9,  0.8,  8.87, 1.51),
    (1600.0, 2.0, 17.0, -16.6,  2.0, 18.95, 4.03),
    (1640.0, 12.0, 26.0,  6.1,  2.8, 18.68, 4.28),
    (1667.0, 9.0, 23.0,  42.7, -1.0,  3.48, 1.11),
    (1673.0, 5.0, 20.0,   1.4,  0.7,  4.67, 0.82),
    (1707.0, 12.0, 16.0,  35.4, -1.0,  1.08, 0.40),
    (1721.0, 5.0, 11.0,  63.6, -1.0,  0.81, 0.36),
    (1739.0, 8.0, 19.0,  42.7, -1.0,  3.44, 1.09),
    (1755.0, 10.0, 17.0,  63.6, -1.0,  1.18, 0.43),
    (1766.0, 4.0,  5.0,  64.0, -1.0,  2.52, 0.75),
    (1783.0, 6.0, 15.0,  64.4, -1.0, 20.81, 7.04),
    (1815.0, 4.0, 10.0,  -8.0,  0.8, 28.08, 4.49),
    (1822.0, 10.0,  8.0,  -7.3, 10.0,  2.02, 0.79),
    (1835.0, 1.0, 20.0,  13.0,  2.0,  9.48, 2.21),
]
eruptions_df = pd.DataFrame(
    ERUPTIONS_RAW, columns=["yearCE", "month", "day", "lat", "hemi", "ssi", "sigma_ssi"],
)

def compute_enso_year_from_month(month: int, year: int) -> int:
    return (year + 1) if (month >= 7) else year
    
eruptions_df["enso_year"] = [
    compute_enso_year_from_month(int(m), int(y))
    for m, y in zip(eruptions_df["month"], eruptions_df["yearCE"])
]

ERUPTION_YEARS = eruptions_df["enso_year"].to_numpy(dtype=int)
 
# Niño3.4 box
LAT_BOUNDS = (-5.0, 5.0)
LON_BOUNDS = (190.0, 240.0)
TROP_LAT_BOUNDS = (-20.0, 20.0)
THRESH = 0.5
EXCLUDE_YEARS = 5
OFFSET = 0
N_MIN_PER_CELL = 1
PHASE_LABELS = {1: "El Niño", 0: "Neutral", -1: "La Niña"}
PHASE_CODES  = [1, 0, -1]
 
 
SAVE_CSV = True
OUT_RAW_PATH  = "/home/563/ft3359/FT-Honours/Honours_Paper/Temporal_Evolution/All_Eruptions/All_eruptions_raw_models.csv"
OUT_CTRL_PATH = "/home/563/ft3359/FT-Honours/Honours_Paper/Temporal_Evolution/All_Eruptions/All_eruptions_control_models.csv"
 
START_YEAR = 850
END_YEAR   = 1849

In [5]:
# HELPERS
 
def classify_phases(values: np.ndarray, thresh: float = 0.5) -> np.ndarray:
    x = np.asarray(values, dtype=float)
    ph = np.zeros_like(x, dtype=int)
    ph[x >=  thresh] =  1
    ph[x <= -thresh] = -1
    return ph
 
def remaining_lifetime(phases: np.ndarray) -> np.ndarray:
    phases = np.asarray(phases, dtype=int)
    n = len(phases)
    rem = np.zeros(n, dtype=int)
    i = 0
    while i < n:
        j = i
        while (j + 1 < n) and (phases[j + 1] == phases[i]):
            j += 1
        for k in range(i, j + 1):
            rem[k] = (j - k + 1)
        i = j + 1
    return rem
 
def eligible_control_mask(years: np.ndarray, eruption_years: np.ndarray, exclude_years: int) -> np.ndarray:
    yrs = np.asarray(years, dtype=int)
    eru = np.asarray(sorted(set(int(y) for y in eruption_years)), dtype=int)
    mask = np.ones_like(yrs, dtype=bool)
    mask &= ~np.isin(yrs, eru)
    for ey in eru:
        mask &= ~((yrs >= ey - exclude_years) & (yrs <= ey + exclude_years))
    return mask
 
 
# Model Relative Niño3.4 Annual ENSO-Year
 
def _to_0_360(lon):
    lon = np.asarray(lon)
    return np.where(lon < 0, lon + 360.0, lon)
 
def _area_weighted_mean_latlon(da: xr.DataArray, latname: str, lonname: str) -> xr.DataArray:
    w = np.cos(np.deg2rad(da[latname]))
    w_da = xr.DataArray(w, coords={latname: da[latname]}, dims=[latname])
    return da.weighted(w_da).mean(dim=(latname, lonname))
 
def compute_relative_nino34_annual_enso_year(ds: xr.Dataset) -> pd.DataFrame:
    vname = "ts" if "ts" in ds.data_vars else list(ds.data_vars)[0]
    latname = "lat" if "lat" in ds.coords else ("latitude" if "latitude" in ds.coords else None)
    lonname = "lon" if "lon" in ds.coords else ("longitude" if "longitude" in ds.coords else None)
    if latname is None or lonname is None:
        raise KeyError("Could not find lat/lon coordinates.")
 
    da = ds[vname]
    lon360 = xr.apply_ufunc(_to_0_360, ds[lonname])
    da = da.assign_coords({lonname: lon360}).sortby(lonname)
 
    sample = float(da.isel(time=0).mean().values)
    if np.isfinite(sample) and sample > 100:
        da = da - 273.15
 
    da_n34 = da.sel(**{latname: slice(LAT_BOUNDS[0], LAT_BOUNDS[1]),
                       lonname: slice(LON_BOUNDS[0], LON_BOUNDS[1])})
    n34 = _area_weighted_mean_latlon(da_n34, latname, lonname)
 
    da_trop = da.sel(**{latname: slice(TROP_LAT_BOUNDS[0], TROP_LAT_BOUNDS[1])})
    trop = _area_weighted_mean_latlon(da_trop, latname, lonname)
 
    if hasattr(n34["time"], "dt"):
        years = n34["time"].dt.year.astype(int).values
        months = n34["time"].dt.month.astype(int).values
    else:
        tvals = n34["time"].values
        years = np.array([int(getattr(t, "year")) for t in tvals], dtype=int)
        months = np.array([int(getattr(t, "month")) for t in tvals], dtype=int)
 
    enso_year = np.array([compute_enso_year_from_month(m, y) for y, m in zip(years, months)], dtype=int)
 
    dfm = pd.DataFrame({
        "enso_year": enso_year,
        "month":     months,
        "n34":       n34.values.astype(float),
        "trop":      trop.values.astype(float),
    }).dropna()
 
    dfm = dfm[(dfm["enso_year"] >= START_YEAR) & (dfm["enso_year"] <= END_YEAR)]
    if dfm.empty:
        return pd.DataFrame(columns=["enso_year", "value"])
 
    # remove monthly climatology
    
    clim_n34  = dfm.groupby("month")["n34"].mean()
    clim_trop = dfm.groupby("month")["trop"].mean()
    dfm["n34_anom"]  = dfm["n34"]  - dfm["month"].map(clim_n34)
    dfm["trop_anom"] = dfm["trop"] - dfm["month"].map(clim_trop)
    dfm["rel"] = dfm["n34_anom"] - dfm["trop_anom"]
 
    # 3-month running mean
    
    dfm["rel"] = dfm["rel"].rolling(window=3, center=True).mean()
 
    # standardise monthly series (per model)
    
    rel_mean = dfm["rel"].mean()
    rel_std  = dfm["rel"].std()
    if not np.isfinite(rel_std) or rel_std == 0.0:
        return pd.DataFrame(columns=["enso_year", "value"])
    dfm["rel"] = (dfm["rel"] - rel_mean) / rel_std
 
    # July-June annual mean
    
    dfa = dfm.groupby("enso_year")["rel"].mean().reset_index()
    dfa = dfa.rename(columns={"rel": "value"}).sort_values("enso_year")
    return dfa

def apply_year_offset(ds: xr.Dataset, year_offset: int) -> xr.Dataset:
    if not year_offset:
        return ds
    shifted = [t.replace(year=t.year - year_offset) for t in ds["time"].values]
    return ds.assign_coords(time=CFTimeIndex(shifted))

In [6]:
# LOAD + PREP PER MODEL
 
per_model = {}
 
for model, cfg in MODEL_PATHS.items():
    path = cfg["path"]
    year_offset = cfg.get("year_offset", 0)
    
    if not os.path.exists(path):
        print(f"Skipping {model} (file not found): {path}")
        continue
 
    ds = xr.open_dataset(path, decode_times=True)
    ds = apply_year_offset(ds, year_offset)
 
    try:
        ann = compute_relative_nino34_annual_enso_year(ds)
    finally:
        ds.close()
 
    if ann.empty:
        print(f"Skipping {model}: no annual data after processing/windowing.")
        continue
 
    years     = ann["enso_year"].astype(int).values
    vals      = ann["value"].astype(float).values
    phases    = classify_phases(vals, THRESH)
    lifetimes = remaining_lifetime(phases)
 
    per_model[model] = dict(
        years=years,
        phases=phases,
        lifetimes=lifetimes,
    )
 
if len(per_model) == 0:
    raise RuntimeError("No models loaded. Check MODEL_PATHS.")
 
 
# BUILD RAW + CONTROL ROWS
 
raw_rows  = []
ctrl_rows = []
 
for phc in PHASE_CODES:
    ph_label = PHASE_LABELS[phc]
 
    for model, data in per_model.items():
        years     = data["years"]
        phases    = data["phases"]
        lifetimes = data["lifetimes"]
 
        year_to_idx = {int(y): i for i, y in enumerate(years)}
 
        # RAW
        for ey in ERUPTION_YEARS:
            ay = int(ey) + OFFSET
            if ay not in year_to_idx:
                continue
            idx = year_to_idx[ay]
            if phases[idx] != phc:
                continue
            lt = float(lifetimes[idx])
            if not np.isfinite(lt):
                continue
            raw_rows.append({
                "source":             "Models",
                "model":              model,
                "Phase":              ph_label,
                "eruption_year":      int(ey),
                "remaining_lifetime": lt,
            })
 
        # CONTROL
        ok_ctrl   = eligible_control_mask(years, ERUPTION_YEARS, EXCLUDE_YEARS)
        ctrl_idxs = [year_to_idx[int(y)] for y in years[(phases == phc) & ok_ctrl]]
        for idx in ctrl_idxs:
            lt = float(lifetimes[idx])
            if not np.isfinite(lt):
                continue
            ctrl_rows.append({
                "source":             "Models",
                "model":              model,
                "Phase":              ph_label,
                "remaining_lifetime": lt,
            })
 
df_raw  = pd.DataFrame(raw_rows)
df_ctrl = pd.DataFrame(ctrl_rows)
 
phase_order = ["El Niño", "Neutral", "La Niña"]
for df in [df_raw, df_ctrl]:
    if not df.empty:
        df["Phase"] = pd.Categorical(df["Phase"], categories=phase_order, ordered=True)
 
df_raw  = df_raw.sort_values(["Phase", "model", "eruption_year"]).reset_index(drop=True)
df_ctrl = df_ctrl.sort_values(["Phase", "model"]).reset_index(drop=True)
 
 
# SAVE
 
if SAVE_CSV:
    os.makedirs(os.path.dirname(OUT_RAW_PATH), exist_ok=True)
    df_raw.to_csv(OUT_RAW_PATH,   index=False)
    df_ctrl.to_csv(OUT_CTRL_PATH, index=False)
    print(f"Saved raw lifetimes : {OUT_RAW_PATH}  ({len(df_raw)} rows)")
    print(f"Saved ctrl lifetimes: {OUT_CTRL_PATH}  ({len(df_ctrl)} rows)")
 

/g/data/xp65/admin/analysis3/sitecustomize.py:72: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  mod = _real_import(name, globals, locals, fromlist, level)
/g/data/xp65/public/apps/med_conda/envs/analysis3-25.10/lib/python3.11/site-packages/xarray/coding/times.py:213: SerializationWarning: Ambiguous reference date string: 850-1-1. The first value is assumed to be the year hence will be padded with zeros to remove the ambiguity (the padded reference date string is: 0850-1-1). To remove this message, remove the ambiguity by padding your reference date strings with zeros.
  ref_date = _ensure_padded_year(ref_date)
/jobfs/178626876.gadi-pbs/ipykernel_3141261/3778320200.py:13: SerializationWarning: Unable to decode time axis into full numpy.datetime64 objects, continuing using cftime.datetime objects instead, reason: dates prior

Saved raw lifetimes : /home/563/ft3359/FT-Honours/Honours_Paper/Temporal_Evolution/All_Eruptions/All_eruptions_raw_models.csv  (200 rows)
Saved ctrl lifetimes: /home/563/ft3359/FT-Honours/Honours_Paper/Temporal_Evolution/All_Eruptions/All_eruptions_control_models.csv  (7999 rows)
